In [2]:
import os
os.chdir("/mnt/xD/Projekty/Python/NewUnoGame/UnoGame/")


In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pandas import DataFrame

from Uno.games_data.dataHandler import prepare_data_for_learning, encode_data, decode_data

In [5]:
df: DataFrame = pd.read_csv("Uno/games_data/20241117_0049_uno_game.csv")
df

,num_red,num_green,num_blue,num_yellow,num_stop,num_plus2,num_plus4,top_card,num_cards_left_in_deck,round,num_enemy_cards,direction,num_colors_in_hand,num_cards_in_hand,is_game_over,index_of_a_player,game_id,card_played,did_win
0,2,2,1,1,0,0,0,All Colors,93,1,7,1,4,7,False,0,e7955a72-a659-448c-9e19-2bb569480ac1,8 Yellow,True
1,3,2,2,0,0,1,0,8 Yellow,93,2,6,1,3,7,False,1,e7955a72-a659-448c-9e19-2bb569480ac1,Draw,False
2,2,2,1,0,0,0,0,8 Yellow,92,3,8,1,3,6,False,0,e7955a72-a659-448c-9e19-2bb569480ac1,All Blue,True
3,4,2,2,0,0,1,0,All Blue,92,4,5,1,3,8,False,1,e7955a72-a659-448c-9e19-2bb569480ac1,3 Blue,False
4,2,2,1,0,0,0,0,3 Blue,92,5,7,1,3,5,False,0,e7955a72-a659-448c-9e19-2bb569480ac1,9 Blue,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5442,0,2,0,0,0,0,0,Stop Yellow,49,238,2,-1,1,2,False,1,4a395b7d-1cc9-422a-8cb9-1d0cf4fcb7db,Draw,True
5443,0,1,0,1,0,0,0,Stop Green,48,239,2,-1,2,2,False,0,4a395b7d-1cc9-422a-8cb9-1d0cf4fcb7db,Stop Stop,False
5444,0,2,0,0,0,0,0,Stop Green,48,240,2,-1,1,2,False,1,4a395b7d-1cc9-422a-8cb9-1d0cf4fcb7db,1 Green,True
5445,0,1,0,1,0,0,0,1 Green,48,241,1,-1,2,2,False,0,4a395b7d-1cc9-422a-8cb9-1d0cf4fcb7db,2 Green,False


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5447 entries, 0 to 5446
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   num_red                 5447 non-null   int64 
 1   num_green               5447 non-null   int64 
 2   num_blue                5447 non-null   int64 
 3   num_yellow              5447 non-null   int64 
 4   num_stop                5447 non-null   int64 
 5   num_plus2               5447 non-null   int64 
 6   num_plus4               5447 non-null   int64 
 7   top_card                5447 non-null   object
 8   num_cards_left_in_deck  5447 non-null   int64 
 9   round                   5447 non-null   int64 
 10  num_enemy_cards         5447 non-null   int64 
 11  direction               5447 non-null   int64 
 12  num_colors_in_hand      5447 non-null   int64 
 13  num_cards_in_hand       5447 non-null   int64 
 14  is_game_over            5447 non-null   bool  
 15  inde

In [48]:
df = prepare_data_for_learning(df)
df, labels = encode_data(df)
df = df.loc[df["card_played"] < 10]

In [49]:
data = np.array(df)
m, n = data.shape # m = cases, n = columns
np.random.shuffle(data)

data = data.T
y_values = data[-1]
x_values = data[:-1]

y_train = y_values[0:round(m*0.75)]
y_test = y_values[round(m*0.75):]
X_train = x_values[:, 0:round(m*0.75)]
X_test = x_values[:, round(m*0.75):]


In [67]:
def initialize_parameters():
    W1 = np.random.randn(10, 16)
    b1 = np.random.randn(10, 1)
    W2 = np.random.randn(10, 10)
    b2 = np.random.randn(10, 1)
    return W1, b1, W2, b2

def ReLu(Z):
    return np.maximum(0, Z)

def softmax(X):
    print(np.exp(X))
    print(np.sum(np.exp(X)))
    return np.exp(X) / np.sum(np.exp(X))

def forward_propagation(X, W1, b1, W2, b2):
    Z1 = W1.dot(X) + b1
    A1 = ReLu(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(A1)
    return Z1, A1, Z2, A2

def one_hot(Y):
    one_hot_Y = np.zeros((Y.shape[0], Y.Max() + 1)) # Creates correct shape output matrix
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def derivative_ReLu(Z):
    return Z > 0

def back_propagation(Z1, A1, Z2, A2, W2, X, Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2, 2)
    dZ1 = W2.T.dot(dZ2) * derivative_ReLu(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1, 2)
    return dW1, db1, dW2, db2

def update_parameters(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2

def get_predictions(A2):
    return np.argmax(A2)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = initialize_parameters()

    for iteration in range(iterations):
        Z1, A1, Z2, A2 = forward_propagation(X, W1, b1, W2, b2)
        dW1, db1, dW2, db2 = back_propagation(Z1, A1, Z2, A2, W2, X, Y)
        W1, b1, W2, B2 = update_parameters(W1, b1, W2, b2, dW2, db2, alpha)
        if iteration % 50 == 0:
            print("Iteration:", iteration)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))
    return W1, b1, W2, b2


In [68]:
W1, b1, W2, b2 = gradient_descent(X_train, y_train, 100, 0.1)


TypeError: loop of ufunc does not support argument 0 of type float which has no callable exp method